# Citations and answer grounding

Exercise notebook for the lesson **[Citations and answer grounding](https://ramkumarmd.com/medical-ai/rag/citations-and-grounding)** in the *Retrieval-augmented generation (RAG)* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** In a RAG system, each statement in an answer should cite the passage it came from, so a reader can check it. But models attach citations to claims their sources do not support, cite the wrong passage, or reverse a rule while keeping its words. Check every claim against its cited source automatically where possible, flag or remove unsupported claims, and remember that simple word-overlap checks miss negation: 'start metformin' and 'do not start metformin' look almost the same to them.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Checking claims against their sources

Claim-level support check against cited sections of the synthetic protocol.

In [ ]:
import re

sections = {f"s{i + 1}": p.strip() for i, p in enumerate(re.split(r"\n(?=## )", PROTOCOL)[1:])}

answer = [   # (claim, section it cites)
    ("Check eGFR at least once a year.", "s2"),
    ("Do not start metformin if eGFR is below 45.", "s2"),
    ("Stop metformin if eGFR falls below 30.", "s4"),          # right fact, wrong citation
    ("Metformin can be continued at any eGFR above 15.", "s2"),  # unsupported claim
    ("Start metformin if eGFR is below 45.", "s2"),             # reverses the rule
]

STOP = {"the", "a", "an", "is", "if", "at", "of", "to", "and", "or", "be", "can", "any", "do", "not"}

def content_words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP}

def support(claim, source_text):
    """Share of the claim's content words, and all of its numbers, found in the cited source."""
    words, src = content_words(claim), content_words(source_text)
    numbers_ok = all(n in src for n in re.findall(r"\d+", claim))
    return len(words & src) / len(words), numbers_ok

for claim, cited in answer:
    overlap, numbers_ok = support(claim, sections[cited])
    verdict = "supported" if overlap >= 0.8 and numbers_ok else "CHECK"
    print(f"{verdict:<9} [{cited}] overlap {overlap:.2f} numbers {'ok' if numbers_ok else 'missing'}  {claim}")

# supported [s2] overlap 1.00 numbers ok  Check eGFR at least once a year.
# supported [s2] overlap 0.80 numbers ok  Do not start metformin if eGFR is below 45.
# CHECK     [s4] overlap 0.17 numbers missing  Stop metformin if eGFR falls below 30.
# CHECK     [s2] overlap 0.40 numbers missing  Metformin can be continued at any eGFR above 15.
# supported [s2] overlap 0.80 numbers ok  Start metformin if eGFR is below 45.

### Repairing citations

Continuing from the previous block: look for a section that does support each flagged claim.

In [ ]:
def best_source(claim):
    return max(sections, key=lambda sid: support(claim, sections[sid])[0])

for claim, cited in answer:
    overlap, numbers_ok = support(claim, sections[cited])
    if not (overlap >= 0.8 and numbers_ok):
        better = best_source(claim)
        o2, n2 = support(claim, sections[better])
        fix = f"cite {better} instead" if o2 >= 0.8 and n2 else "no section supports it: remove or flag"
        print(f"{claim}\n  -> {fix}")

# Stop metformin if eGFR falls below 30.
#   -> cite s2 instead
# Metformin can be continued at any eGFR above 15.
#   -> no section supports it: remove or flag

## Exercise

**Goal:** Build and stress-test a claim-level citation checker on the synthetic protocol.

**Data:** Synthetic protocol and answer claims in the notebook (Created for this course; not clinical guidance)

### Step 1

Reproduce the support check and the citation repair.

In [ ]:
# Your code here


### Step 2

Write ten more claims, including three negated or reversed ones, and record which the checker gets wrong.

In [ ]:
# Your code here


### Step 3

Add a negation rule: flag any claim whose negation words ('not', 'no', 'stop', 'never') differ from the best-matching source sentence.

In [ ]:
# Your code here


### Step 4

Optional: try an open natural language inference model on the same claims and compare.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/rag/citations-and-grounding](https://ramkumarmd.com/medical-ai/rag/citations-and-grounding). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.